# Machine Learning — Assignment 2

## Classification Model Comparison and Streamlit Application

**Name:** Thoolika CA 
**BITS ID:** 2026AK05165
**Programme:** M.Tech Artificial Intelligence and Machine Learning  
**Institution:** BITS Pilani — WILP  

In [2]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

# Load the dataset as a table
cancer_dataset = load_breast_cancer(as_frame=True)
dataset = cancer_dataset.frame.copy()

# Check the assignment's dataset requirements
print("Number of instances:", dataset.shape[0])
print("Number of input features:", cancer_dataset.data.shape[1])
print("Target labels:", cancer_dataset.target_names)
print("Target encoding: 0 = malignant, 1 = benign")

dataset.head()

Number of instances: 569
Number of input features: 30
Target labels: ['malignant' 'benign']
Target encoding: 0 = malignant, 1 = benign


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [11]:
%pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 119.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 161.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 143.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Check data quality
print("Total missing values:", dataset.isna().sum().sum())
print("Duplicate rows:", dataset.duplicated().sum())

# Count samples in each class
print("\nSamples in each class:")
class_counts = dataset["target"].value_counts().sort_index()
class_counts.index = ["Malignant (0)", "Benign (1)"]
print(class_counts)

# Confirm the minimum dataset requirements
assert dataset.shape[0] >= 500, "At least 500 instances required"
assert cancer_dataset.data.shape[1] >= 12, "At least 12 features required"
print("\nDataset size requirements satisfied.")

Total missing values: 0
Duplicate rows: 0

Samples in each class:
Malignant (0)    212
Benign (1)       357
Name: count, dtype: int64

Dataset size requirements satisfied.


In [6]:
from sklearn.model_selection import train_test_split

# Separate input features from the target
X = dataset.drop(columns=["target"])
y = dataset["target"]

# Preserve approximately the same class proportions in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training targets:", y_train.shape)
print("Testing targets:", y_test.shape)

# Confirm that training and testing rows do not overlap
assert set(X_train.index).isdisjoint(set(X_test.index))
print("\nNo overlap between training and testing rows.")

Training features: (455, 30)
Testing features: (114, 30)
Training targets: (455,)
Testing targets: (114,)

No overlap between training and testing rows.


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

# Store models and results for the later comparison
trained_models = {}
evaluation_results = []

# Reuse this function to evaluate all five models consistently
def evaluate_model(model_name, model):
    predictions = model.predict(X_test)

    # Select the probability of malignant, whose label is 0
    malignant_column = list(model.classes_).index(0)
    malignant_probability = model.predict_proba(X_test)[:, malignant_column]

    result = {
        "ML Model Name": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "AUC": roc_auc_score(
            (y_test == 0).astype(int), malignant_probability
        ),
        "Precision": precision_score(
            y_test, predictions, pos_label=0, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, pos_label=0, zero_division=0
        ),
        "F1": f1_score(
            y_test, predictions, pos_label=0, zero_division=0
        ),
        "MCC": matthews_corrcoef(y_test, predictions)
    }
    return result

# Scale features and train Logistic Regression
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

logistic_model.fit(X_train, y_train)

trained_models["Logistic Regression"] = logistic_model
evaluation_results.append(
    evaluate_model("Logistic Regression", logistic_model)
)

pd.DataFrame(evaluation_results).round(4)

,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9825,0.9954,0.9762,0.9762,0.9762,0.9623


In [10]:
from sklearn.tree import DecisionTreeClassifier

# Train using the same training data
decision_tree_model = DecisionTreeClassifier(random_state=42)
decision_tree_model.fit(X_train, y_train)

# Store the trained model
trained_models["Decision Tree"] = decision_tree_model

# Replace any previous result for this model if the cell is rerun
evaluation_results = [
    result for result in evaluation_results
    if result["ML Model Name"] != "Decision Tree"
]

# Calculate all six metrics on the same test data
evaluation_results.append(
    evaluate_model("Decision Tree", decision_tree_model)
)

pd.DataFrame(evaluation_results).round(4)

,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9825,0.9954,0.9762,0.9762,0.9762,0.9623
1,Decision Tree,0.9123,0.9157,0.8478,0.9286,0.8864,0.8174


In [12]:
from sklearn.neighbors import KNeighborsClassifier

# Scale features because KNN uses distances between samples
knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", KNeighborsClassifier(n_neighbors=5))
])

knn_model.fit(X_train, y_train)

# Store the trained model
trained_models["KNN"] = knn_model

# Avoid duplicate results if this cell is rerun
evaluation_results = [
    result for result in evaluation_results
    if result["ML Model Name"] != "KNN"
]

# Calculate all six metrics using the same test set
evaluation_results.append(
    evaluate_model("KNN", knn_model)
)

pd.DataFrame(evaluation_results).round(4)

,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9825,0.9954,0.9762,0.9762,0.9762,0.9623
1,Decision Tree,0.9123,0.9157,0.8478,0.9286,0.8864,0.8174
2,KNN,0.9561,0.9788,0.9512,0.9286,0.9398,0.9054


In [14]:
from sklearn.naive_bayes import GaussianNB

# Train Gaussian Naive Bayes
naive_bayes_model = GaussianNB()
naive_bayes_model.fit(X_train, y_train)

# Store the trained model
trained_models["Naive Bayes"] = naive_bayes_model

# Avoid duplicate results if this cell is rerun
evaluation_results = [
    result for result in evaluation_results
    if result["ML Model Name"] != "Naive Bayes"
]

# Calculate all six metrics using the same test set
evaluation_results.append(
    evaluate_model("Naive Bayes", naive_bayes_model)
)

pd.DataFrame(evaluation_results).round(4)

,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9825,0.9954,0.9762,0.9762,0.9762,0.9623
1,Decision Tree,0.9123,0.9157,0.8478,0.9286,0.8864,0.8174
2,KNN,0.9561,0.9788,0.9512,0.9286,0.9398,0.9054
3,Naive Bayes,0.9386,0.9878,0.9268,0.9048,0.9157,0.8676


In [16]:
from sklearn.svm import SVC

# Scale features and train an SVM with an RBF kernel
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    ))
])

svm_model.fit(X_train, y_train)

# Store the trained model
trained_models["Support Vector Machine"] = svm_model

# Avoid duplicate results if this cell is rerun
evaluation_results = [
    result for result in evaluation_results
    if result["ML Model Name"] != "Support Vector Machine"
]

# Calculate all six metrics using the same test set
evaluation_results.append(
    evaluate_model("Support Vector Machine", svm_model)
)

comparison_table = pd.DataFrame(evaluation_results)
comparison_table.round(4)

/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9825,0.9954,0.9762,0.9762,0.9762,0.9623
1,Decision Tree,0.9123,0.9157,0.8478,0.9286,0.8864,0.8174
2,KNN,0.9561,0.9788,0.9512,0.9286,0.9398,0.9054
3,Naive Bayes,0.9386,0.9878,0.9268,0.9048,0.9157,0.8676
4,Support Vector Machine,0.9825,0.9950,0.9762,0.9762,0.9762,0.9623


In [18]:
from pathlib import Path
import joblib
import json

# Create the required model folder
model_folder = Path("model")
model_folder.mkdir(exist_ok=True)

model_files = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "svm.joblib"
}

# Save all five models, including scaling where used
for model_name, filename in model_files.items():
    joblib.dump(trained_models[model_name], model_folder / filename)
    print("Saved:", filename)

# Save only the held-out test data, with its target column
test_data = X_test.copy()
test_data["target"] = y_test
test_data.to_csv("test_data.csv", index=False)

# Save the comparison results
comparison_table.to_csv("model_comparison.csv", index=False)

# Record the input columns and class meanings for the app
metadata = {
    "feature_names": X_train.columns.tolist(),
    "target_column": "target",
    "target_mapping": {"0": "malignant", "1": "benign"},
    "positive_class": 0,
    "model_files": model_files
}

Path("metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

print("\nSaved test_data.csv:", test_data.shape)
print("Saved model_comparison.csv and metadata.json")

Saved: logistic_regression.joblib
Saved: decision_tree.joblib
Saved: knn.joblib
Saved: naive_bayes.joblib
Saved: svm.joblib

Saved test_data.csv: (114, 31)
Saved model_comparison.csv and metadata.json


In [20]:
from IPython.display import display

print("Machine Learning - Assignment 2")
print("Name: Thoolika CA")
print("BITS ID: 2026AK05165")
print("Dataset: Breast Cancer Wisconsin (Diagnostic)")
print("Training samples:", len(X_train))
print("Test samples:", len(X_test))
print("Positive class for Precision, Recall, F1 and AUC: malignant (0)")
print("\nEvaluation of all five models:")
display(comparison_table.round(4))

Machine Learning - Assignment 2
Name: Thoolika CA
BITS ID: 2026AK05165
Dataset: Breast Cancer Wisconsin (Diagnostic)
Training samples: 455
Test samples: 114
Positive class for Precision, Recall, F1 and AUC: malignant (0)

Evaluation of all five models:


,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9825,0.9954,0.9762,0.9762,0.9762,0.9623
1,Decision Tree,0.9123,0.9157,0.8478,0.9286,0.8864,0.8174
2,KNN,0.9561,0.9788,0.9512,0.9286,0.9398,0.9054
3,Naive Bayes,0.9386,0.9878,0.9268,0.9048,0.9157,0.8676
4,Support Vector Machine,0.9825,0.9950,0.9762,0.9762,0.9762,0.9623


In [22]:
import ast
from pathlib import Path

app_source = Path("app.py").read_text(encoding="utf-8")
ast.parse(app_source)

print("app.py syntax check passed.")

app.py syntax check passed.


In [24]:
%pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 kB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 149.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 131.9 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.3
    Uninstalling protobuf-4.25.3:
      Successfully uninstalled protobuf-4.25.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.17.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 7.

In [26]:
import subprocess
import sys

# Launch Streamlit in a separate Python process
app_log = open("streamlit.log", "w")

app_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.port=8501",
        "--server.headless=true"
    ],
    stdout=app_log,
    stderr=subprocess.STDOUT
)

print("Streamlit launch requested.")
print("Open http://localhost:8501 in the lab's Firefox browser.")

Streamlit launch requested.
Open http://localhost:8501 in the lab's Firefox browser.


In [28]:
from importlib.metadata import version
from pathlib import Path

# Record the installed versions used by this application
packages = [
    "streamlit",
    "scikit-learn",
    "numpy",
    "pandas",
    "scipy",
    "joblib",
    "Jinja2"
]

requirements_text = "\n".join(
    f"{package}=={version(package)}"
    for package in packages
) + "\n"

Path("requirements.txt").write_text(
    requirements_text,
    encoding="utf-8"
)

print("requirements.txt created:\n")
print(requirements_text)

requirements.txt created:

streamlit==1.67.0
scikit-learn==1.9.1
numpy==2.5.4
pandas==2.2.3
scipy==1.18.1
joblib==1.4.2
Jinja2==3.1.4



In [30]:
verification_rows = []

for model_name, model in trained_models.items():
    predictions = model.predict(X_test)

    correct = int((predictions == y_test.to_numpy()).sum())
    total = len(y_test)

    verification_rows.append({
        "Model": model_name,
        "Correct predictions": correct,
        "Wrong predictions": total - correct,
        "Total test samples": total,
        "Accuracy (%)": round(correct / total * 100, 2)
    })

display(pd.DataFrame(verification_rows))

,Model,Correct predictions,Wrong predictions,Total test samples,Accuracy (%)
0,Logistic Regression,112,2,114,98.25
1,Decision Tree,104,10,114,91.23
2,KNN,109,5,114,95.61
3,Naive Bayes,107,7,114,93.86
4,Support Vector Machine,112,2,114,98.25


In [32]:
from pathlib import Path
print(Path.cwd())

/home/s2026ak05165/workspace/ML_Assignment_2
